In [1]:
import numpy as np
import scipy
import scipy.integrate as integrate
import scipy.interpolate as interpolate
import matplotlib.pyplot as plt
import math
import pickle
from matplotlib import cm
from matplotlib.ticker import LinearLocator
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import MaxNLocator
import matplotlib.tri as tri
from scipy.interpolate import RegularGridInterpolator
import matplotlib.colors as colors


In [11]:
#Ni Elastic tensor
C_voigt = np.array([[276, 159, 159, 0, 0, 0],
                    [159, 276, 159, 0, 0, 0],
                    [159, 159, 276, 0, 0, 0],
                    [0, 0, 0, 132, 0, 0],
                    [0, 0, 0, 0, 132, 0],
                    [0, 0, 0, 0, 0, 132]]) / 10**3
Voigt2Tensor = np.array([[1, 6, 5],
                        [6, 2, 4],
                        [5, 4, 3]], dtype=int) - 1
C = np.zeros((3,3,3,3))
for i in range(3):
    for j in range(3):
        for k in range(3):
            for l in range(3):
                C[i,j,k,l] = C_voigt[Voigt2Tensor[i,j],Voigt2Tensor[k,l]]

In [16]:
#Anisotropy integral
#####THIS WORKS
lebedev_quad = np.loadtxt('lebedev_131.txt')
def M_func(C, ksi_vec):
    #print(ksi_vec)
    M = np.einsum('ikjl,k,l->ij',C,ksi_vec,ksi_vec)
    M_star = np.linalg.inv(M)
    return M_star
def P_integral_total(l,m,C,Ms):
    num_of_points = len(lebedev_quad[:,0])
    res = np.zeros((3,3),dtype=np.complex_)
    for i in range(0,num_of_points):
        w = lebedev_quad[i,2]
        
        point = np.array([np.cos(np.radians(lebedev_quad[i,0])) * np.sin(np.radians(lebedev_quad[i,1])),\
                      np.sin(np.radians(lebedev_quad[i,0])) * np.sin(np.radians(lebedev_quad[i,1])),\
                      np.cos(np.radians(lebedev_quad[i,1]))])
        
        theta = np.radians(lebedev_quad[i,0])
        if theta < 0:
            theta = theta + 2 * np.pi
        
        phi = np.radians(lebedev_quad[i,1])
        
        Y_bar = np.conjugate(scipy.special.sph_harm(m,l,theta,phi))
        res = res + w * Ms[:,:,i] * Y_bar * 4 * np.pi #4pi is due to lebedev. This is cartesian lebedev
    return res


#calculate the Ms
num_of_points = len(lebedev_quad[:,0])
Ms = np.zeros((3,3,num_of_points))
for i in range(0,num_of_points):
    w = lebedev_quad[:,2]
    point = np.array([np.cos(np.radians(lebedev_quad[i,0])) * np.sin(np.radians(lebedev_quad[i,1])),\
                      np.sin(np.radians(lebedev_quad[i,0])) * np.sin(np.radians(lebedev_quad[i,1])),\
                      np.cos(np.radians(lebedev_quad[i,1]))])
    #print(M_func(C,point).shape)
    Ms[:,:,i] = M_func(C,point)

l_max = 20
for l in range(0,l_max + 1,2):
    for m in range(0, l + 1):
        print('Calculating l = {}, m = {}'.format(int(l), int(m)))
        P = np.zeros((3,3),dtype=np.complex_)
        P = P_integral_total(l,m,C,Ms[:,:,:])
        print('Resulting P:\n',np.linalg.det(P))
        with open('A/A_{}_{}.pkl'.format(int(l), int(m)), 'wb') as outp:
            pickle.dump(P, outp, pickle.HIGHEST_PROTOCOL)

Calculating l = 0, m = 0
Resulting P:
 (23327.798973318502+0j)
Calculating l = 2, m = 0
Resulting P:
 (-3.9320805643038512+0j)
Calculating l = 2, m = 1
Resulting P:
 (-6.0624137335676084e-15-3.2274192894975683e-16j)
Calculating l = 2, m = 2
Resulting P:
 (-7.801879238654683e-15-2.2526910431922907e-15j)
Calculating l = 4, m = 0
Resulting P:
 (-6.363942067687732+0j)
Calculating l = 4, m = 1
Resulting P:
 (-4.3405115518058476e-18+9.672934283708747e-18j)
Calculating l = 4, m = 2
Resulting P:
 (-3.6748072828844653e-16-1.2471377216289173e-16j)
Calculating l = 4, m = 3
Resulting P:
 (-3.311204834562863e-17+3.3949051904318965e-17j)
Calculating l = 4, m = 4
Resulting P:
 (1.7194019474140427+2.530558078138203e-15j)
Calculating l = 6, m = 0
Resulting P:
 (0.058430864047344995+0j)
Calculating l = 6, m = 1
Resulting P:
 (-7.135225763757622e-17-6.591107937298781e-18j)
Calculating l = 6, m = 2
Resulting P:
 (6.4527777566061436e-18-2.0394419437127873e-18j)
Calculating l = 6, m = 3
Resulting P:
 (2.265

In [22]:
l_max = 20

for l in range(0,l_max + 1,2):
    for m in range(0,l+1):
        print('l: {} m: {}'.format(l,m))
        with open('A/A_{}_{}.pkl'.format(int(l),int(m)), 'rb') as f:
            P = pickle.load(f)
        np.savetxt('A_matrices/A_{}_{}_r.txt'.format(int(l),int(m)),P.real)
        np.savetxt('A_matrices/A_{}_{}_i.txt'.format(int(l),int(m)),P.imag)

l: 0 m: 0
l: 2 m: 0
l: 2 m: 1
l: 2 m: 2
l: 4 m: 0
l: 4 m: 1
l: 4 m: 2
l: 4 m: 3
l: 4 m: 4
l: 6 m: 0
l: 6 m: 1
l: 6 m: 2
l: 6 m: 3
l: 6 m: 4
l: 6 m: 5
l: 6 m: 6
l: 8 m: 0
l: 8 m: 1
l: 8 m: 2
l: 8 m: 3
l: 8 m: 4
l: 8 m: 5
l: 8 m: 6
l: 8 m: 7
l: 8 m: 8
l: 10 m: 0
l: 10 m: 1
l: 10 m: 2
l: 10 m: 3
l: 10 m: 4
l: 10 m: 5
l: 10 m: 6
l: 10 m: 7
l: 10 m: 8
l: 10 m: 9
l: 10 m: 10
l: 12 m: 0
l: 12 m: 1
l: 12 m: 2
l: 12 m: 3
l: 12 m: 4
l: 12 m: 5
l: 12 m: 6
l: 12 m: 7
l: 12 m: 8
l: 12 m: 9
l: 12 m: 10
l: 12 m: 11
l: 12 m: 12
l: 14 m: 0
l: 14 m: 1
l: 14 m: 2
l: 14 m: 3
l: 14 m: 4
l: 14 m: 5
l: 14 m: 6
l: 14 m: 7
l: 14 m: 8
l: 14 m: 9
l: 14 m: 10
l: 14 m: 11
l: 14 m: 12
l: 14 m: 13
l: 14 m: 14
l: 16 m: 0
l: 16 m: 1
l: 16 m: 2
l: 16 m: 3
l: 16 m: 4
l: 16 m: 5
l: 16 m: 6
l: 16 m: 7
l: 16 m: 8
l: 16 m: 9
l: 16 m: 10
l: 16 m: 11
l: 16 m: 12
l: 16 m: 13
l: 16 m: 14
l: 16 m: 15
l: 16 m: 16
l: 18 m: 0
l: 18 m: 1
l: 18 m: 2
l: 18 m: 3
l: 18 m: 4
l: 18 m: 5
l: 18 m: 6
l: 18 m: 7
l: 18 m: 8
l: 18 m: 9
l: 18 m:

In [20]:
np.linalg.norm(point)

1.0